In [1]:
import pandas as pd
import numpy as np
import warnings, gc, os
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from IPython.display import display
pd.set_option('display.max_columns', None)
!rm -rf /kaggle/working/*

In [2]:
train = pd.read_csv("/kaggle/input/binary-classification-with-a-bank-database/train.csv")
test  = pd.read_csv("/kaggle/input/binary-classification-with-a-bank-database/test.csv")
train.drop(columns=["id"],axis=1,inplace=True)

print("Check Out Train DaTA Null Values: ",train.isnull().sum())
print(f"Train Data Shape: {train.shape}")
print(f"Train Data INFO: {train.info()}")

def bank_feature_engineering(df):
    df = df.copy()
  
    df['no_previous_contact'] = (df['pdays'].isna()).astype(int)
    df['ever_contacted']      = 1 - df['no_previous_contact']
    df['duration_hour'] = df['duration'] // 60
    df['duration_min']  = df['duration'] % 60
    df['is_long_call']  = (df['duration'] > 500).astype(int)
    df['is_very_long_call'] = (df['duration'] > 900).astype(int)
    df['is_young'] = (df['age'] <= 30).astype(int)
    df['is_senior'] = (df['age'] >= 60).astype(int)
    df['age_x_balance'] = df['age'] * df['balance'].clip(lower=0)
    df['has_balance'] = (df['balance'] > 0).astype(int)
    df['high_balance'] = (df['balance'] > 3000).astype(int)
    df['negative_balance'] = (df['balance'] < 0).astype(int)
    df['was_contacted_before'] = (df['previous'] > 0).astype(int)
    df['many_campaigns'] = (df['campaign'] > 4).astype(int)
    df['previous_per_campaign'] = df['previous'] / (df['campaign'] + 1)
    month_order = {'jan':1, 'feb':2, 'mar':3, 'apr':4, 'may':5, 'jun':6,
                   'jul':7, 'aug':8, 'sep':9, 'oct':10, 'nov':11, 'dec':12}
    df['month_num'] = df['month'].map(month_order)
    df['is_quarter_end'] = df['month'].isin(['mar', 'jun', 'sep', 'dec']).astype(int)
    df['job_education'] = df['job'] + "_" + df['education']
    df['job_marital']   = df['job'] + "_" + df['marital']
    df['balance_bin']   = pd.qcut(df['balance'], q=10, duplicates='drop').astype(str)
    df['poutcome_success_before'] = (df['poutcome'] == 'success').astype(int)
    df['poutcome_failure_before'] = (df['poutcome'] == 'failure').astype(int)
    df['contact_month'] = df['contact'] + "_" + df['month']
    df['debt_burden'] = df['housing'].apply(lambda x: 1 if x=='yes' else 0) + df['loan'].apply(lambda x: 1 if x=='yes' else 0)
    df['call_in_best_month'] = df['month'].isin(['mar', 'sep', 'oct', 'dec']).astype(int)
    df['student_or_retired'] = df['job'].isin(['student', 'retired']).astype(int)
    return df


train = bank_feature_engineering(train)
test=bank_feature_engineering(test)

balance_bins = [-10000, 0, 500, 1000, 3000, 10000, 100000]  # adjust based on your dataset
balance_labels = ['neg', 'very_low', 'low', 'medium', 'high', 'very_high']
train['balance_bin'] = pd.cut(train['balance'], bins=balance_bins, labels=balance_labels, include_lowest=True)
train['balance_bin'] = train['balance_bin'].astype(str)

test['balance_bin'] = pd.cut(test['balance'], bins=balance_bins, labels=balance_labels, include_lowest=True)
# test['balance_bin'] = test['balance_bin'].astype(str)


categorical_cols = train.select_dtypes(include='object').columns.tolist()

if 'y' in categorical_cols:
    categorical_cols.remove('y')  


label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
  
    test[col] = le.transform(test[col])  
    
    label_encoders[col] = le


numeric_cols = ['age','balance','duration','campaign','previous','age_x_balance']
scaler = StandardScaler()
train[numeric_cols] = scaler.fit_transform(train[numeric_cols])
test[numeric_cols] = scaler.transform(test[numeric_cols])
Id=test.id
test.drop(columns=["id"],axis=1,inplace=True)

print("Display Train Data:")
display(train.head())
print("#"*130)
print("\n")
print("Display Test Data:")
display(test.head())

Check Out Train DaTA Null Values:  age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64
Train Data Shape: (750000, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 17 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   age        750000 non-null  int64 
 1   job        750000 non-null  object
 2   marital    750000 non-null  object
 3   education  750000 non-null  object
 4   default    750000 non-null  object
 5   balance    750000 non-null  int64 
 6   housing    750000 non-null  object
 7   loan       750000 non-null  object
 8   contact    750000 non-null  object
 9   day        750000 non-null  int64 
 10  month      750000 non-null  object
 11  duration   750000 non-null  in

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y,no_previous_contact,ever_contacted,duration_hour,duration_min,is_long_call,is_very_long_call,is_young,is_senior,age_x_balance,has_balance,high_balance,negative_balance,was_contacted_before,many_campaigns,previous_per_campaign,month_num,is_quarter_end,job_education,job_marital,balance_bin,poutcome_success_before,poutcome_failure_before,contact_month,debt_burden,call_in_best_month,student_or_retired
0,0.106310,9,1,1,0,-0.422083,0,0,0,25,1,-0.510829,0.155597,-1,-0.223475,3,0,0,1,1,57,0,0,0,0,-0.394414,1,0,0,0,0,0.0,8,0,37,28,5,0,0,1,0,0,0
1,-0.289776,1,1,1,0,-0.243316,0,0,2,18,6,-0.261338,-0.580100,-1,-0.223475,3,0,0,1,3,5,0,0,0,0,-0.250238,1,0,0,0,0,0.0,6,1,5,4,1,0,0,30,0,0,0
2,-0.487819,1,1,1,0,-0.212287,1,0,2,14,8,-0.532843,-0.212251,-1,-0.223475,3,0,0,1,1,51,0,0,0,0,-0.234201,1,0,0,0,0,0.0,5,0,5,4,1,0,0,32,1,0,0
3,-1.379012,8,2,1,0,-0.412563,1,0,2,28,8,-0.903409,-0.212251,-1,-0.223475,3,0,0,1,0,10,0,0,1,0,-0.389737,1,0,0,0,0,0.0,5,0,33,26,5,0,0,32,1,0,1
4,-1.478033,9,1,1,0,-0.111092,1,0,0,3,3,2.369319,-0.580100,-1,-0.223475,3,1,0,1,15,2,1,1,1,0,-0.223394,1,0,0,0,0,0.0,2,0,37,28,1,0,0,3,1,0,0


##################################################################################################################################


Display Test Data:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,no_previous_contact,ever_contacted,duration_hour,duration_min,is_long_call,is_very_long_call,is_young,is_senior,age_x_balance,has_balance,high_balance,negative_balance,was_contacted_before,many_campaigns,previous_per_campaign,month_num,is_quarter_end,job_education,job_marital,balance_bin,poutcome_success_before,poutcome_failure_before,contact_month,debt_burden,call_in_best_month,student_or_retired
0,-0.883905,1,1,1,0,0.068028,1,0,2,21,8,-0.118248,-0.580100,-1,-0.223475,3,0,1,3,44,0,0,0,0,-0.061592,1,0,0,0,0,0.0,5,0,5,4,2,0,0,32,1,0,0
1,0.304353,4,1,2,0,-0.416441,1,0,0,3,0,1.209922,-0.212251,-1,-0.223475,3,0,1,9,46,1,0,0,0,-0.389033,1,0,0,0,0,0.0,4,0,18,13,5,0,0,0,1,0,0
2,-0.487819,6,1,0,0,-0.408332,1,1,0,13,8,-0.532843,-0.212251,-1,-0.223475,3,0,1,1,51,0,0,0,0,-0.384206,1,0,0,0,0,0.0,5,0,24,19,5,0,0,8,2,0,0
3,1.690653,1,1,1,0,-0.911136,1,1,2,29,8,-0.481477,-0.580100,-1,-0.223475,3,0,1,2,5,0,0,0,0,-0.396617,0,0,1,0,0,0.0,5,0,5,4,3,0,0,32,2,0,0
4,-1.279990,9,2,1,0,0.263014,1,0,0,22,5,-0.276014,-0.580100,-1,-0.223475,3,0,1,3,1,0,0,1,0,0.012571,1,0,0,0,0,0.0,7,0,37,29,2,0,0,5,1,0,0


In [3]:
features = [c for c in train.columns if c not in ['id', 'y']]

params_lgb = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 1e-3,
    'num_leaves': 128,
    'feature_fraction': 0.75,
    'bagging_fraction': 0.85,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42,
    'max_depth': -1,
    'min_data_in_leaf': 100,
    'lambda_l1': 0.1,
    'lambda_l2': 0.5,
    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0
}

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train))
test_preds_lgb = np.zeros(len(test))

for fold, (trn_idx, val_idx) in enumerate(skf.split(train, train['y'])):
    print(f"\nFold {fold+1}/10")
    X_trn, y_trn = train.iloc[trn_idx][features], train.iloc[trn_idx]['y']
    X_val, y_val = train.iloc[val_idx][features], train.iloc[val_idx]['y']
    
    dtrain = lgb.Dataset(X_trn, y_trn)
    dvalid = lgb.Dataset(X_val, y_val, reference=dtrain)
    
    model = lgb.train(params_lgb,dtrain,num_boost_round=5000,valid_sets=[dtrain, dvalid],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)])
    
    oof_preds[val_idx] = model.predict(X_val)
    test_preds_lgb += model.predict(test[features]) / skf.n_splits
    
    print(f"Fold {fold+1} AUC: {roc_auc_score(y_val, oof_preds[val_idx]):.6f}")

print(f"\nOverall OOF AUC: {roc_auc_score(train['y'], oof_preds):.6f}")



Fold 1/10


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Training until validation scores don't improve for 100 rounds
[200]	training's auc: 0.959589	valid_1's auc: 0.96029
[400]	training's auc: 0.960291	valid_1's auc: 0.960903
[600]	training's auc: 0.96083	valid_1's auc: 0.961398
[800]	training's auc: 0.961321	valid_1's auc: 0.961853
[1000]	training's auc: 0.96177	valid_1's auc: 0.962257
[1200]	training's auc: 0.962169	valid_1's auc: 0.962617
[1400]	training's auc: 0.96258	valid_1's auc: 0.962976
[1600]	training's auc: 0.962983	valid_1's auc: 0.96334
[1800]	training's auc: 0.963422	valid_1's auc: 0.963723
[2000]	training's auc: 0.963867	valid_1's auc: 0.964105
[2200]	training's auc: 0.964301	valid_1's auc: 0.964466
[2400]	training's auc: 0.964693	valid_1's auc: 0.964787
[2600]	training's auc: 0.965089	valid_1's auc: 0.965125
[2800]	training's auc: 0.965472	valid_1's auc: 0.965448
[3000]	training's auc: 0.965814	valid_1's auc: 0.965734
[3200]	training's auc: 0.966155	valid_1's auc: 0.966015
[3400]	training's auc: 0.966481	valid_1's auc: 0.96

In [4]:
params_cb = {
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'learning_rate': 1e-3,
    'iterations': 5000,
    'depth': 8,
    'l2_leaf_reg': 3,
    'random_strength': 0.8,
    'bagging_temperature': 0.7,
    'random_seed': 42,
    'task_type': 'GPU',
    'devices': '0',
    'early_stopping_rounds': 400,
    'verbose': 500
}

test_preds_cb = np.zeros(len(test))
oof_preds_cb = np.zeros(len(train))

for fold, (trn_idx, val_idx) in enumerate(skf.split(train, train['y'])):
    print(f"\nCAT Fold {fold+1}/10")
    
    X_trn = train.iloc[trn_idx][features].values
    X_val = train.iloc[val_idx][features].values
    y_trn = train.iloc[trn_idx]['y'].values
    y_val = train.iloc[val_idx]['y'].values
    
    model = cb.CatBoost(params_cb)
    model.fit(X_trn, y_trn, eval_set=(X_val, y_val), use_best_model=True, verbose=500)
    
    oof_preds_cb[val_idx] = model.predict(X_val, prediction_type='Probability')[:, 1]
    test_preds_cb += model.predict(test[features].values, prediction_type='Probability')[:, 1] / skf.n_splits

print(f"\nCatBoost OOF AUC: {roc_auc_score(train['y'], oof_preds_cb):.6f}")



CAT Fold 1/10


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9416739	best: 0.9416739 (0)	total: 264ms	remaining: 21m 59s
500:	test: 0.9491093	best: 0.9491709 (433)	total: 7.72s	remaining: 1m 9s
1000:	test: 0.9504738	best: 0.9504738 (1000)	total: 14.9s	remaining: 59.6s
1500:	test: 0.9524667	best: 0.9524667 (1500)	total: 22.3s	remaining: 52s
2000:	test: 0.9542561	best: 0.9542561 (2000)	total: 29.8s	remaining: 44.7s
2500:	test: 0.9558064	best: 0.9558064 (2500)	total: 37.4s	remaining: 37.3s
3000:	test: 0.9571573	best: 0.9571573 (3000)	total: 45s	remaining: 30s
3500:	test: 0.9582304	best: 0.9582304 (3500)	total: 52.6s	remaining: 22.5s
4000:	test: 0.9591169	best: 0.9591169 (4000)	total: 1m	remaining: 15s
4500:	test: 0.9598452	best: 0.9598452 (4500)	total: 1m 7s	remaining: 7.5s
4999:	test: 0.9604698	best: 0.9604698 (4999)	total: 1m 15s	remaining: 0us
bestTest = 0.960469842
bestIteration = 4999

CAT Fold 2/10


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9386275	best: 0.9386275 (0)	total: 25.9ms	remaining: 2m 9s
500:	test: 0.9476463	best: 0.9476874 (450)	total: 7.69s	remaining: 1m 9s
1000:	test: 0.9489518	best: 0.9489518 (1000)	total: 15.2s	remaining: 1m
1500:	test: 0.9508435	best: 0.9508435 (1500)	total: 22.8s	remaining: 53.2s
2000:	test: 0.9526047	best: 0.9526047 (2000)	total: 30.6s	remaining: 45.9s
2500:	test: 0.9541557	best: 0.9541557 (2500)	total: 38.4s	remaining: 38.4s
3000:	test: 0.9555417	best: 0.9555417 (3000)	total: 46.1s	remaining: 30.7s
3500:	test: 0.9566715	best: 0.9566715 (3500)	total: 53.8s	remaining: 23s
4000:	test: 0.9575286	best: 0.9575286 (4000)	total: 1m 1s	remaining: 15.3s
4500:	test: 0.9582663	best: 0.9582663 (4500)	total: 1m 8s	remaining: 7.64s
4999:	test: 0.9589009	best: 0.9589009 (4999)	total: 1m 16s	remaining: 0us
bestTest = 0.9589008689
bestIteration = 4999

CAT Fold 3/10


...

0:	test: 0.9411466	best: 0.9411466 (0)	total: 26.2ms	remaining: 2m 11s
500:	test: 0.9484401	best: 0.9484401 (500)	total: 7.69s	remaining: 1m 9s
1000:	test: 0.9498461	best: 0.9498472 (998)	total: 15.2s	remaining: 1m
1500:	test: 0.9518078	best: 0.9518078 (1500)	total: 22.8s	remaining: 53.1s
2000:	test: 0.9535570	best: 0.9535570 (2000)	total: 30.6s	remaining: 45.8s
2500:	test: 0.9550375	best: 0.9550375 (2500)	total: 38.4s	remaining: 38.3s
3000:	test: 0.9563038	best: 0.9563038 (3000)	total: 46.1s	remaining: 30.7s
3500:	test: 0.9573554	best: 0.9573554 (3500)	total: 53.8s	remaining: 23s
4000:	test: 0.9582058	best: 0.9582058 (4000)	total: 1m 1s	remaining: 15.3s
4500:	test: 0.9589043	best: 0.9589043 (4500)	total: 1m 9s	remaining: 7.66s
4999:	test: 0.9594749	best: 0.9594749 (4999)	total: 1m 16s	remaining: 0us
bestTest = 0.9594748616
bestIteration = 4999

CAT Fold 9/10


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9399146	best: 0.9399146 (0)	total: 25.8ms	remaining: 2m 9s
500:	test: 0.9484386	best: 0.9484954 (377)	total: 7.66s	remaining: 1m 8s
1000:	test: 0.9498748	best: 0.9498748 (1000)	total: 15.1s	remaining: 1m
1500:	test: 0.9517769	best: 0.9517769 (1500)	total: 22.7s	remaining: 53s
2000:	test: 0.9534405	best: 0.9534405 (2000)	total: 30.4s	remaining: 45.6s
2500:	test: 0.9548990	best: 0.9548990 (2500)	total: 38.2s	remaining: 38.2s
3000:	test: 0.9561738	best: 0.9561738 (3000)	total: 46s	remaining: 30.6s
3500:	test: 0.9572783	best: 0.9572783 (3500)	total: 53.6s	remaining: 23s
4000:	test: 0.9581421	best: 0.9581421 (4000)	total: 1m 1s	remaining: 15.3s
4500:	test: 0.9588584	best: 0.9588584 (4500)	total: 1m 8s	remaining: 7.64s
4999:	test: 0.9594773	best: 0.9594773 (4999)	total: 1m 16s	remaining: 0us
bestTest = 0.9594773054
bestIteration = 4999

CAT Fold 10/10


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9381632	best: 0.9381632 (0)	total: 25.9ms	remaining: 2m 9s
500:	test: 0.9465535	best: 0.9465728 (463)	total: 7.64s	remaining: 1m 8s
1000:	test: 0.9479845	best: 0.9479845 (1000)	total: 15.1s	remaining: 1m
1500:	test: 0.9500884	best: 0.9500884 (1500)	total: 22.7s	remaining: 52.9s
2000:	test: 0.9519092	best: 0.9519092 (2000)	total: 30.4s	remaining: 45.6s
2500:	test: 0.9534443	best: 0.9534443 (2500)	total: 38.2s	remaining: 38.2s
3000:	test: 0.9548417	best: 0.9548417 (3000)	total: 45.9s	remaining: 30.6s
3500:	test: 0.9559921	best: 0.9559921 (3500)	total: 53.6s	remaining: 22.9s
4000:	test: 0.9568647	best: 0.9568647 (4000)	total: 1m 1s	remaining: 15.3s
4500:	test: 0.9576181	best: 0.9576181 (4500)	total: 1m 8s	remaining: 7.63s
4999:	test: 0.9582739	best: 0.9582739 (4999)	total: 1m 16s	remaining: 0us
bestTest = 0.9582738876
bestIteration = 4999

CatBoost OOF AUC: 0.959074


In [5]:
best_auc = 0
best_w = None

for w in np.linspace(0, 1, 101):
    blend = w * oof_preds + (1-w) * oof_preds_cb
    auc = roc_auc_score(train['y'], blend)
    if auc > best_auc:
        best_auc = auc
        best_w = w

print("Best weight:", best_w, "AUC:", best_auc)


Best weight: 1.0 AUC: 0.9670073380026379


In [6]:
final_pred = 0.86 * test_preds_lgb + 0.14 * test_preds_cb
submission=pd.DataFrame({"id":Id,"y":final_pred})
submission.to_csv('submission.csv', index=False)
print("Submission saved!")
submission.head(5)

Submission saved!


,id,y
0,750000,0.008008
1,750001,0.132966
2,750002,0.001925
3,750003,0.001417
4,750004,0.028028
